In [ ]:
# Authentication: run `earthengine authenticate` once and set the environment
# variable EE_PROJECT to your own Google Cloud project before executing.
import os
# -*- coding: utf-8 -*-
"""Quantifica_faltantes_RSEI_multiplos_assets.py

Script para Colab/GEE que lê múltiplos assets de origem, agrega os registros
e quantifica anos faltantes por município (NM_MUN), mantendo a lógica original
do fluxo de auditoria.
"""

# ================================================================
# LISTAR IMAGENS E IDENTIFICAR ANOS FALTANTES POR NM_MUN NO GEE
#
# Assets de leitura:
# 1) projects/YOUR_PROJECT/assets/RSEI_Bands_2026_new_new
# 2) projects/YOUR_PROJECT_2/assets/RSEI_Bands_2026_new_new
#
# O script:
# 1) Inicializa o Earth Engine no Colab
# 2) Tenta ler cada asset como ImageCollection
# 3) Caso não funcione, tenta ler como pasta/coleção via ee.data.listAssets
# 4) Lista todas as imagens com asset_origem, asset_id, NM_MUN e ANO
# 5) Consolida os registros dos dois assets
# 6) Verifica, para cada NM_MUN, quais anos existem entre 1985 e 2025
# 7) Gera a lista de anos faltantes por município
# 8) Salva os resultados em CSV/XLSX
# ================================================================

# =========================
# 1. INSTALAR E IMPORTAR
# =========================
!pip -q install earthengine-api pandas openpyxl

import ee
import pandas as pd
from google.colab import drive

# =========================
# 2. AUTENTICAÇÃO
# =========================
try:
    ee.Initialize(project=os.environ.get('EE_PROJECT'))
except Exception:
    ee.Authenticate()
    # ee.Initialize(project=os.environ.get('EE_PROJECT'))
    ee.Initialize(project=os.environ.get('EE_PROJECT'))

# Opcional: montar Google Drive para salvar os CSVs
MONTAR_DRIVE = True
if MONTAR_DRIVE:
    drive.mount('/content/drive')

# =========================
# 3. CONFIGURAÇÕES
# =========================
ASSET_IDS_LEITURA = [
    'projects/YOUR_PROJECT/assets/RSEI_Bands_2026_new_new',
    'projects/YOUR_PROJECT_2/assets/RSEI_Bands_2026_new_new'
]

ANO_INICIAL = 1985
ANO_FINAL = 2025
ANOS_ESPERADOS = list(range(ANO_INICIAL, ANO_FINAL + 1))

# Pasta de saída
OUT_DIR = '<YOUR_DRIVE_PATH>' if MONTAR_DRIVE else '/content'



In [ ]:
# =========================
# 4. FUNÇÕES AUXILIARES
# =========================
def normalizar_ano(valor):
    """
    Converte o valor do campo ANO para inteiro, quando possível.
    Retorna None se não conseguir converter.
    """
    if valor is None:
        return None
    try:
        return int(valor)
    except Exception:
        try:
            return int(float(valor))
        except Exception:
            return None

def normalizar_nm_mun(valor):
    """
    Garante string para NM_MUN.
    """
    if valor is None:
        return None
    return str(valor).strip()

def listar_assets_em_pasta(asset_id):
    """
    Lista assets contidos em uma pasta/coleção usando ee.data.listAssets.
    Funciona para casos em que o asset não puder ser lido diretamente
    como ImageCollection.
    """
    resultado = ee.data.listAssets({'parent': asset_id})
    assets = resultado.get('assets', [])
    return assets

def tentar_ler_como_imagecollection(asset_id):
    """
    Tenta ler diretamente como ImageCollection e retornar metadados
    agregados de forma eficiente.
    """
    ic = ee.ImageCollection(asset_id)
    n = ic.size().getInfo()  # força avaliação
    return ic, n

def obter_registros_via_imagecollection(asset_id):
    """
    Lê o asset como ImageCollection e retorna uma lista de registros:
    asset_origem, asset_id, system:index, NM_MUN, ANO
    """
    ic = ee.ImageCollection(asset_id)

    fc = ic.map(
        lambda img: ee.Feature(
            None,
            {
                'asset_origem': asset_id,
                'asset_id': img.get('system:id'),
                'system_index': img.get('system:index'),
                'NM_MUN': img.get('NM_MUN'),
                'ANO': img.get('ANO')
            }
        )
    )

    registros = fc.getInfo()['features']
    saida = []
    for f in registros:
        props = f['properties']
        saida.append({
            'asset_origem': props.get('asset_origem', asset_id),
            'asset_id': props.get('asset_id'),
            'system_index': props.get('system_index'),
            'NM_MUN': normalizar_nm_mun(props.get('NM_MUN')),
            'ANO': normalizar_ano(props.get('ANO'))
        })
    return saida

def obter_registros_via_listassets(asset_id):
    """
    Caso o asset seja uma pasta com imagens, lista os filhos e lê as
    propriedades NM_MUN e ANO de cada imagem.
    """
    assets = listar_assets_em_pasta(asset_id)

    registros = []
    for item in assets:
        child_id = item.get('id')
        child_type = item.get('type')

        if child_type != 'IMAGE':
            continue

        try:
            img = ee.Image(child_id)
            props = img.toDictionary(['NM_MUN', 'ANO', 'system:index', 'system:id']).getInfo()

            registros.append({
                'asset_origem': asset_id,
                'asset_id': props.get('system:id', child_id),
                'system_index': props.get('system:index'),
                'NM_MUN': normalizar_nm_mun(props.get('NM_MUN')),
                'ANO': normalizar_ano(props.get('ANO'))
            })
        except Exception as e:
            print(f'Erro ao ler propriedades de {child_id}: {e}')

    return registros

def ler_registros_de_um_asset(asset_id):
    """
    Lê um asset individual, tentando primeiro como ImageCollection e,
    se necessário, como listagem de assets.
    """
    print(f'Tentando ler o asset: {asset_id}')
    print('-' * 70)

    try:
        _, n = tentar_ler_como_imagecollection(asset_id)
        print(f'Asset lido como ImageCollection com {n} imagens.')
        registros = obter_registros_via_imagecollection(asset_id)
        return registros
    except Exception as e:
        print('Não foi possível ler diretamente como ImageCollection.')
        print(f'Motivo: {e}')
        print('Tentando leitura como pasta/listagem de assets...')

        registros = obter_registros_via_listassets(asset_id)
        print(f'Foram encontrados {len(registros)} assets do tipo IMAGE.')
        return registros

def gerar_tabelas_de_saida(registros, anos_esperados):
    """
    A partir da lista de registros, gera:
    1) tabela detalhada de imagens
    2) tabela resumida por NM_MUN com anos existentes e faltantes
    3) tabela expandida com um ano faltante por linha
    """
    df = pd.DataFrame(registros)

    if df.empty:
        raise ValueError('Nenhum registro foi encontrado nos assets informados.')

    # Remover duplicados exatos entre os dois assets, preservando a primeira ocorrência
    colunas_chave = ['NM_MUN', 'ANO', 'system_index', 'asset_id']
    colunas_existentes = [c for c in colunas_chave if c in df.columns]
    if colunas_existentes:
        df = df.drop_duplicates(subset=colunas_existentes).copy()

    df = df.sort_values(
        by=['NM_MUN', 'ANO', 'asset_origem', 'system_index'],
        na_position='last'
    ).reset_index(drop=True)

    resumo = []
    faltantes_expandido = []

    for nm_mun, grupo in df.groupby('NM_MUN', dropna=False):
        anos_existentes = sorted({a for a in grupo['ANO'].tolist() if a is not None})
        anos_faltantes = [a for a in anos_esperados if a not in anos_existentes]

        assets_origem = sorted({a for a in grupo['asset_origem'].dropna().tolist()})

        resumo.append({
            'NM_MUN': nm_mun,
            'n_imagens': len(grupo),
            'n_assets_origem': len(assets_origem),
            'assets_origem': ' | '.join(assets_origem),
            'anos_existentes': ', '.join(map(str, anos_existentes)) if anos_existentes else '',
            'qtd_anos_existentes': len(anos_existentes),
            'anos_faltantes': ', '.join(map(str, anos_faltantes)) if anos_faltantes else '',
            'qtd_anos_faltantes': len(anos_faltantes),
            'completo_1985_2025': 'SIM' if len(anos_faltantes) == 0 else 'NAO'
        })

        for ano in anos_faltantes:
            faltantes_expandido.append({
                'NM_MUN': nm_mun,
                'ANO_FALTANTE': ano
            })

    df_resumo = pd.DataFrame(resumo).sort_values(by=['NM_MUN']).reset_index(drop=True)
    df_faltantes_expandido = pd.DataFrame(faltantes_expandido)

    if not df_faltantes_expandido.empty:
        df_faltantes_expandido = df_faltantes_expandido.sort_values(
            by=['NM_MUN', 'ANO_FALTANTE']
        ).reset_index(drop=True)

    return df, df_resumo, df_faltantes_expandido

# =========================
# 5. LEITURA DOS ASSETS
# =========================
registros = []

for asset_id in ASSET_IDS_LEITURA:
    registros_asset = ler_registros_de_um_asset(asset_id)
    print(f'Total de registros lidos em {asset_id}: {len(registros_asset)}')
    print('=' * 70)
    registros.extend(registros_asset)

print(f'Total bruto de registros somando todos os assets: {len(registros)}')



In [ ]:
# =========================
# 6. GERAR TABELAS
# =========================
df_imagens, df_resumo, df_faltantes = gerar_tabelas_de_saida(registros, ANOS_ESPERADOS)

# =========================
# 7. EXIBIR RESULTADOS
# =========================
print('\n' + '=' * 70)
print('TABELA DETALHADA DAS IMAGENS')
print('=' * 70)
print(df_imagens.head(20))

print('\n' + '=' * 70)
print('RESUMO POR NM_MUN')
print('=' * 70)
print(df_resumo.head(20))

print('\n' + '=' * 70)
print('ANOS FALTANTES POR NM_MUN')
print('=' * 70)
print(df_faltantes.head(50))

# =========================
# 8. SALVAR CSVs
# =========================
import os
os.makedirs(OUT_DIR, exist_ok=True)

csv_imagens = f'{OUT_DIR}/rsei_imagens_listadas_multiplos_assets.csv'
csv_resumo = f'{OUT_DIR}/rsei_resumo_anos_por_municipio_multiplos_assets.csv'
csv_faltantes = f'{OUT_DIR}/rsei_anos_faltantes_por_municipio_multiplos_assets.csv'
xlsx_saida = f'{OUT_DIR}/rsei_auditoria_anos_multiplos_assets.xlsx'

df_imagens.to_csv(csv_imagens, index=False, encoding='utf-8-sig')
df_resumo.to_csv(csv_resumo, index=False, encoding='utf-8-sig')
df_faltantes.to_csv(csv_faltantes, index=False, encoding='utf-8-sig')

with pd.ExcelWriter(xlsx_saida, engine='openpyxl') as writer:
    df_imagens.to_excel(writer, sheet_name='imagens', index=False)
    df_resumo.to_excel(writer, sheet_name='resumo_municipios', index=False)
    df_faltantes.to_excel(writer, sheet_name='anos_faltantes', index=False)

print('\n' + '=' * 70)
print('ARQUIVOS SALVOS COM SUCESSO')
print('=' * 70)
print(csv_imagens)
print(csv_resumo)
print(csv_faltantes)
print(xlsx_saida)

# =========================
# 9. RELATÓRIO FINAL NO CONSOLE
# =========================
total_municipios = df_resumo.shape[0]
municipios_completos = (df_resumo['completo_1985_2025'] == 'SIM').sum()
municipios_incompletos = (df_resumo['completo_1985_2025'] == 'NAO').sum()
total_imagens = df_imagens.shape[0]
total_faltantes = df_faltantes.shape[0]

print('\n' + '=' * 70)
print('RELATÓRIO FINAL')
print('=' * 70)
print(f'Total de assets de leitura: {len(ASSET_IDS_LEITURA)}')
print(f'Total de imagens listadas (consolidado): {total_imagens}')
print(f'Total de municípios (NM_MUN): {total_municipios}')
print(f'Municípios completos (1985-2025): {municipios_completos}')
print(f'Municípios com anos faltantes: {municipios_incompletos}')
print(f'Total de registros de anos faltantes: {total_faltantes}')

# =========================
# 10. OPCIONAL: MOSTRAR SOMENTE MUNICÍPIOS INCOMPLETOS
# =========================
df_incompletos = df_resumo[df_resumo['completo_1985_2025'] == 'NAO'].copy()

print('\n' + '=' * 70)
print('MUNICÍPIOS COM ANOS FALTANTES')
print('=' * 70)
print(df_incompletos[['NM_MUN', 'qtd_anos_faltantes', 'anos_faltantes']].head(50))
